<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/junchao/Program_B(new).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

In [ ]:
import pandas as pd
from google.colab import files

# 1. 读取您上传的两个文件
# 假设文件名分别为 'nodeExport.txt' 和 'x-beam.txt'
# 如果您的文件名不同，请修改这里
df_all = pd.read_csv('nodeExport.txt', sep='\t')
df_ids = pd.read_csv('x-beam.txt', sep='\t')

# 2. 清理列名（去除多余空格）并统一格式
# 提取坐标文件中的 ID, X, Y, Z (假设列名包含 "Node", "X", "Y", "Z")
df_all = df_all[['Node Number', 'X Location (m)', 'Y Location (m)', 'Z Location (m)']]
df_all.columns = ['Node', 'X', 'Y', 'Z']

# 提取 ID 文件中的 Node 列
df_ids = df_ids[['Node Number']]
df_ids.columns = ['Node']

# 3. 合并数据 (类似于 Excel 的 VLOOKUP)
#只保留 df_ids 中存在的节点
result = pd.merge(df_ids, df_all, on='Node', how='left')

# 4. 保存为 TXT 文件
output_filename = 'matched_nodes_coordinates.txt'
result.to_csv(output_filename, sep='\t', index=False)
print(f"文件已生成: {output_filename}")

# 5. 自动触发下载 (Colab 特有功能)
files.download(output_filename)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# 读取数据
df = pd.read_csv('matched_nodes_coordinates.txt', sep='\t')

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# 计算各轴的范围
x_range = df['X'].max() - df['X'].min()
y_range = df['Y'].max() - df['Y'].min()
z_range = df['Z'].max() - df['Z'].min()

# 设置绘图点，颜色设为粉色 (hotpink)
ax.scatter(df['X'], df['Y'], df['Z'], c='hotpink', marker='.', s=5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Point Cloud - True Scale (Pink)')

# 【关键步骤】设置坐标轴比例一致
# 这一步告诉 matplotlib 按照数据的真实比例来显示盒子
ax.set_box_aspect((x_range, y_range, z_range))

# 调整视角 (俯视)
ax.view_init(elev=90, azim=-90)

plt.savefig('3d_point_cloud_pink_true_scale.png')
plt.show()

# Setup

In [ ]:
import os
import sys
import itertools
import functools

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

from pathlib import Path

In [ ]:
pd.set_option('display.max_columns', 100)

## Data getting (if on Colab)


In [ ]:
import google.colab
google.colab.drive.mount("/content/drive")

Get ancillary data from Github

In [ ]:
!git clone https://github.com/marius-ne/CIE_ProjectB_Group13.git

## Load data

In [ ]:
os.chdir("CIE_ProjectB_Group13")

In [ ]:
os.getcwd()

**IMPORTANT:** You need to add a shortcut of the "programB" folder on GoogleDrive to your own "MyDrive" for this to work

In [ ]:
!ln -s /content/drive/MyDrive/programB data

In [ ]:
target_folder = "data/Data2"
current_folder = os.getcwd()

if Path(current_folder).name != target_folder:
    os.chdir(Path(current_folder) / Path(target_folder))
print(os.getcwd())


Convert data

In [ ]:
VARIABLES = {
      0: "TotalDeformation",
      1: "DirectionalDeformation_X_axis",
      2: "DirectionalDeformation_Y_axis",
      3: "DirectionalDeformation_Z_axis",
      4: "EquivalentStress",
      5: "ShearStress_XY",
      6: "ShearStress_XZ",
      7: "ShearStress_YZ",
  }
LOADS = {
    0: "Bigger_train",
    1: "Smaller_train",
}
SEASONS = {
    0: "Summer",
    1: "Winter",
}
HEALTHS = {
    0: "Perfect_structure",
    1: "ip_frst_Arc_defect_all_tracks_111",
    2: "ip_1and3track_3_arc_78910",
    3: "ip_first_track_3arc_78910",
    4: "Ip_1track_1_arc_345",
    5: "ip_3track_1_arc_678",
    6: "ip_2_arc_all_tracks_222",
}
TRAIN_CONFIGS = {
    0: "One_train_1st_track",
    1: "One_train_middle_track",
    2: "Two_trains_extreme_track_different_direction",
    3: "Two_trains_extreme_track_same_direction",
}
VARIABLE_NAMES = list(VARIABLES.values())
NODE_NUMBERS = None

def combination_to_string(combination):
    train_config, load, season, health, variable = combination
    return f"{TRAIN_CONFIGS[train_config]}__{LOADS[load]}__{SEASONS[season]}__{HEALTHS[health]}__{VARIABLE_NAMES[variable]}"

# Construct list of scenarios (combinations of train configs, loads, seasons, healths, variables)
#   Each scenario is a tuple of (train_config, load, season, health, variable), each encoded
#   as the corresponding key in the dictionaries above
combinations = itertools.product(
        TRAIN_CONFIGS.keys(),
        LOADS.keys(),
        SEASONS.keys(),
        HEALTHS.keys(),
        VARIABLES.keys()
    )
combinations = list(combinations)

# Group by variables, i.e. each group has all variables for one scenario
combinations_grouped_by_variable = [
    combinations[i:i + len(VARIABLES)] for i in range(0, len(combinations), len(VARIABLES))
]
# Group further by healths, i.e. each group has all healths for one scenario (train config, load, season)
combinations_grouped_by_health = [
    combinations_grouped_by_variable[i:i + len(HEALTHS)] for i in range(0, len(combinations_grouped_by_variable), len(HEALTHS))
]



In [ ]:
perfect_combinations = [c for c in combinations if c[3] == 6 ]
ipfirstarcdefectalltracks_combinations = [c for c in combinations if c[3] == 5 ]
perfect_combinations, ipfirstarcdefectalltracks_combinations

In [ ]:
combinations_grouped_by_health[0]

In [ ]:
def read_data_file(
    train_config: int = 0,
    load: int = 0,
    season: int = 0,
    health: int = 0,
    variable: int = 0,
):
  """Reads data according to format and provides the data-frame as-is, with
  the categorical variables added as columns."""

  # Construct filename from scenario according to the folder structure

  results_paths = ["Results", "Results1"]

  for results_path in results_paths:
      filename = Path()
      filename /= TRAIN_CONFIGS[train_config]
      filename /= LOADS[load]
      filename /= SEASONS[season]
      filename /= HEALTHS[health]
      filename /= results_path
      filename /= VARIABLES[variable] + ".csv"

      if filename.exists():
          break
  else:
      raise FileNotFoundError(f"Data file not found for combination: {combination_to_string((train_config, load, season, health, variable))}")

  # Encode the scenario as categorical columns
  #   -> TODO: Is there a way of encoding that
  #   preserves information? E.g. like encoding the name of a city as its latitude
  df = pd.read_csv(filename)
  num_nodes = len(df)
  df["season"] = season*np.ones(num_nodes,dtype=np.uint8)
  df["health"] = health*np.ones(num_nodes,dtype=np.uint8)
  df["load"] = load*np.ones(num_nodes,dtype=np.uint8)
  df["train_config"] = train_config*np.ones(num_nodes,dtype=np.uint8)

  return df

def get_data_variable_aggregated(
    scenario_combination
):
    global NODE_NUMBERS

    # Create all possible variable-combinations for that scenario
    combinations_grouped_by_variable = [
       [*scenario_combination,i] for i in VARIABLES.keys()
    ]

    dfs_variables = []
    #  Merge data for all variables in the current health scenario
    for same_variable_combination in combinations_grouped_by_variable:
      print("Processing combination:", combination_to_string(same_variable_combination))

      var_name = VARIABLE_NAMES[same_variable_combination[-1]]

      # Get data file for current combination
      df = read_data_file(*same_variable_combination)

      # Turn the variable column into a single one and add a new time column
      df_melted = df.melt(id_vars=["Node Number","season","load","health","train_config"],var_name="variable",value_name=var_name)
      df_melted["time"] = df_melted["variable"].str[-3:].astype(np.float64)
      df_melted.drop(columns=["variable"],inplace=True)

      # Get node numbers and ensure they're consistent
      if NODE_NUMBERS is None:
        NODE_NUMBERS = df_melted["Node Number"].unique()
      else:
        try:
          assert all(NODE_NUMBERS == df_melted["Node Number"].unique())
        except ValueError or AssertionError:
          print("WARNING: Node numbers differ between data files!")
          print("Previous node numbers:", NODE_NUMBERS)
          print("Current node numbers:", df_melted["Node Number"].unique())

      dfs_variables.append(df_melted)

    # Concatenating all variables into a single data frame
    # -> we do an OUTER join, meaning all keys are kept (A U B)
    #   this should be safe, node numbers and the other shared columns are kept
    shared_cols = ["Node Number","season","load","health","train_config","time"]
    df_vars = functools.reduce(lambda left,right: pd.merge(left,right,on=shared_cols,
                                              how='outer'), dfs_variables)
    # Check that data has been preserved
    for df in dfs_variables:
      for var_name in VARIABLE_NAMES:
        if var_name in df.columns:
          merged = pd.merge(df[shared_cols + [var_name]], df_vars[shared_cols + [var_name]],
                            on=shared_cols, how='inner')
          assert len(merged) == len(df)

    return df_vars


def get_data_variable_and_health_aggregated(
    scenario_combination: tuple
):
    """
    Reads all data files from one scenario of load, train_config and season and takes all health groups
    and variables and merges them into a single data-frame.

    Args:
        scenario_combination (tuple): A tuple of (train_config, load, season) representing
            the scenario for which to aggregate data across all healths and variables.
    Returns:
        pd.DataFrame: Merged data-frame with all variables as columns.
    """
    global NODE_NUMBERS

    # Generate all health and variable combinations for the given (train_config, load, season)
    train_config, load, season = scenario_combination
    combinations_grouped_by_health = [
      [
        [train_config, load, season, health, variable]
        for variable in VARIABLES.keys()
      ]
      for health in HEALTHS.keys()
    ]

    # Merge data for all healths in the scenario
    dfs_healths = []
    for same_health_combinations in combinations_grouped_by_health:

        df_vars = get_data_variable_aggregated(same_health_combinations[0][:-1])

        dfs_healths.append(df_vars)

    # Check that columns are the same
    assert all(all(df.columns == dfs_healths[0].columns) for df in dfs_healths)

    # Concatenate them together
    df = pd.concat(dfs_healths,ignore_index=True)

    return df

def get_data_all_aggregated():
    """
    Reads all data files for all combinations of season, load, train_config, health, and variable,
    and merges them into a single DataFrame.
    Returns:
      pd.DataFrame: Merged data-frame with all variables as columns for all scenarios.
    """
    all_dfs = []
    # Iterate over all combinations of season, load, train_config, and health
    for train_config in TRAIN_CONFIGS.keys():
        for load in LOADS.keys():
            for season in SEASONS.keys():
                for health in HEALTHS.keys():
                    scenario = (train_config, load, season, health)
                    # get_data_variable_aggregated expects a 4-tuple (train_config, load, season, health)
                    try:
                        df_vars = get_data_variable_aggregated(scenario)
                        all_dfs.append(df_vars)
                    except FileNotFoundError as e:
                        print(f"Skipping missing file for scenario: {combination_to_string((*scenario, 0))}")
                        continue
    if not all_dfs:
        raise RuntimeError("No data files found for any scenario.")
    # Concatenate all scenarios together
    df_all = pd.concat(all_dfs, ignore_index=True)
    return df_all

df = get_data_variable_and_health_aggregated([0,0,1,])

In [ ]:
# TODO: save this dataframe as a file
# get_data_all_aggregated()

In [ ]:
df

# Analyze node numbers

Go through all data files and check number of nodes. Create csv file that recaps these.

In [ ]:
nums = []
for ix, combination in enumerate(combinations):
    num_nodes = len(read_data_file(*combination)["Node Number"].unique())
    entry = {}
    entry["train_config"] = combination[0]
    entry["load"] = combination[1]
    entry["season"] = combination[2]
    entry["health"] = combination[3]
    entry["variable"] = combination[4]
    entry["num_nodes"] = num_nodes
    nums.append(entry)
nums

Exemplary plot of node numbers throughout a number of scenarios.

In [ ]:
nn_df = pd.DataFrame(nums)
nn_df.plot(
    x="health", y="num_nodes", kind="bar",
    title=f"Num. nodes per health state"
)

In [ ]:
# Save recap to file
# nn_df.to_csv("node_numbers_Data2_overview.csv", index=False)

# Visualize bridge structure

In [ ]:
# Read tab seperated node export file
node_xyz = pd.read_csv(
    "nodeExport.txt",
    sep="\t",
    engine="python"
)

In [ ]:
def select_df_subset(combination):
    """Selects a subset of the main data-frame according to the given combination.

    Args:
        combination (tuple): A tuple of (train_config, load, season, health, variable).
    """
    train_config, load, season, health, variable = combination
    var_name = VARIABLE_NAMES[variable]
    df_subset = df[
        (df["train_config"] == train_config) &
        (df["load"] == load) &
        (df["season"] == season) &
        (df["health"] == health)
    ][["Node Number", "time", var_name]]
    return df_subset

In [ ]:
from ipywidgets import interact, FloatSlider

# Colors for missing nodes
missing_color = 'r'
missing_nn = {}
colors = [
    missing_color if nn in missing_nn.values() else 'b'
    for nn in node_xyz["Node Number"]
]

cmap = plt.get_cmap('viridis')

# Colors for bridge loads
combination = (0, 0, 0, 3, 0)  # Example combination

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

def plot_bridge_3d_structure(highlight_nodes=None, color_scale: dict=None, s: int=4):
    """
    Plots the bridge structure in 3D using Plotly.
    Optionally highlights nodes in highlight_nodes, or uses a color scale if provided.
    Args:
        highlight_nodes (list, optional): List of node numbers to highlight.
        color_scale (dictionary-like, optional): Dictionary of {node_number: value}
            of values bewteen 0 and 1 to use for coloring nodes.
    """
    if color_scale is not None:
        # color_scale is a dict: {node_number: value}, nodes not present get value 0
        node_numbers = node_xyz["Node Number"].values
        node_colors = [color_scale.get(nn, np.nan) for nn in node_numbers]
        fig = go.Figure(data=[go.Scatter3d(
            x=node_xyz["X Location (m)"],
            y=node_xyz["Y Location (m)"],
            z=node_xyz["Z Location (m)"],
            mode='markers',
            marker=dict(
                size=s,
                color=node_colors,
                colorscale='Viridis',
                colorbar=dict(title="Value"),
                showscale=True,
            )
        )])
        fig.update_layout(
            title="Bridge Structure (colored by value)",
            scene=dict(
                xaxis_title='X Location (m)',
                yaxis_title='Y Location (m)',
                zaxis_title='Z Location (m)'
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )
    else:
        # Default color for all nodes
        colors = ['blue'] * len(node_xyz)
        # Highlight specified nodes
        if highlight_nodes is not None:
            node_indices = node_xyz[node_xyz["Node Number"].isin(highlight_nodes)].index
            for idx in node_indices:
                colors[idx] = 'red'

        fig = go.Figure(data=[go.Scatter3d(
            x=node_xyz["X Location (m)"],
            y=node_xyz["Y Location (m)"],
            z=node_xyz["Z Location (m)"],
            mode='markers',
            marker=dict(
                size=s,
                color=colors,
            )
        )])
        fig.update_layout(
            title="Bridge Structure (highlighted nodes in red)",
            scene=dict(
                xaxis_title='X Location (m)',
                yaxis_title='Y Location (m)',
                zaxis_title='Z Location (m)'
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )
    fig.show()

def plot_bridge_3d_load(combination, time_point):
    """
    Plots bridge loads in 3D for a single time point.
    Args:
        combination (tuple): (train_config, load, season, health, variable)
        time_point (float): Time value to plot
    """
    df_subset = select_df_subset(combination)
    variable_to_plot = VARIABLE_NAMES[combination[-1]]
    timestamp_subset = df_subset[df_subset["time"] == time_point]
    node_loads = [
        timestamp_subset[timestamp_subset["Node Number"] == nn][variable_to_plot].values
        for nn in node_xyz["Node Number"]
    ]
    node_loads_flat = [nl[0] if len(nl) > 0 else np.nan for nl in node_loads]

    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(
        node_xyz["X Location (m)"],
        node_xyz["Y Location (m)"],
        node_xyz["Z Location (m)"],
        c=node_loads_flat, marker='o', s=2,
        cmap="viridis"
    )
    fig.colorbar(scatter, shrink=0.5)
    ax.set_title(f"Bridge Loads at time {time_point}\nFor combination: {combination_to_string(combination)}")
    plt.show()

def plot_bridge_loads_3d_slider(df):
    """
    Plots bridge loads in 3D with a time slider.
    Args:
        df (pd.DataFrame): Dataframe to visualize.
    """
    time_points = np.sort(df["time"].unique())
    variable_to_plot = VARIABLE_NAMES[combination[-1]]

    def plot_at_time(time_point_index):
        time_point = time_points[int(time_point_index)]
        timestamp_subset = df[df["time"] == time_point]
        node_loads = [
            timestamp_subset[timestamp_subset["Node Number"] == nn][variable_to_plot].values
            for nn in node_xyz["Node Number"]
        ]
        node_loads_flat = [nl[0] if len(nl) > 0 else np.nan for nl in node_loads]

        fig = plt.figure(figsize=(10,10))
        ax = fig.add_subplot(111, projection='3d')
        scatter = ax.scatter(
            node_xyz["X Location (m)"],
            node_xyz["Y Location (m)"],
            node_xyz["Z Location (m)"],
            c=node_loads_flat, marker='o', s=2,
            cmap="viridis"
        )
        fig.colorbar(scatter, shrink=0.5)
        ax.set_title(f"Bridge Loads at time {time_point}\nFor combination: {combination_to_string(combination)}")
        fig.savefig(f"../visualization/bridge_loads_3d_{time_point}.png")
        # fig.show()

    interact(
        plot_at_time,
        time_point_index=FloatSlider(
            min=0,
            max=len(time_points)-1,
            step=1,
            value=0,
            description='Time Index'
        )
    )

plot_bridge_loads_3d_slider(df_impfct)

Create GIF from visualizations

In [ ]:
import imageio
import os

# Get all PNG files in the visualization directory, sorted by filename
image_dir = "../visualization/"
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".png")])


# Read images and create GIF
images = [imageio.imread(os.path.join(image_dir, fname)) for fname in image_files]
gif_path = os.path.join(image_dir, "bridge_loads_animation.gif")
imageio.mimsave(gif_path, images, duration=500)

print(f"GIF saved to {gif_path}")

# Remove NaN

In [ ]:
df.dropna(inplace=True)
df

# Data Inspection

In [ ]:
df.dtypes

In [ ]:
m = 5
df.iloc[:m*10].plot(subplots=True,figsize=(15,15))

In [ ]:
df_sorted = df.sort_values(by=["Node Number","health","time"], ascending=[True, True, True])
# Attention - make sure the indices are reset after sorting
df_sorted.reset_index(drop=True, inplace=True)
df_sorted

In [ ]:
df_sorted[["Node Number","health","time","EquivalentStress"]].iloc[:70].plot(subplots=True,figsize=(10,5))

# Find imperfect nodes

In [ ]:
config1 = [0,0,0,1]
config2 = [0,0,0,0]
df1 = get_data_variable_aggregated(config1)
df2 = get_data_variable_aggregated(config2)
df1

In [ ]:
def get_df_differences(combination1, combination2):
  """
  Args:
    comnbination1: (train_config, load, season, health)
    combination2: (train_config, load, season, health)
  """
  df1 = get_data_variable_aggregated(config1)
  df2 = get_data_variable_aggregated(config2)
  df_diff = df1.copy()
  df_diff[VARIABLE_NAMES] = (df2-df1)[VARIABLE_NAMES]
  return df_diff

df_diff = df1.copy()
df_diff[VARIABLE_NAMES] = (df2-df1)[VARIABLE_NAMES]
# Imperfect nodes: all nodes where at least one variable differs between healthy and unhealthy
df_impfct = df_diff[~np.isclose(df_diff[VARIABLE_NAMES], 0, atol=100).all(axis=1)]
# Normalize the variables to between 0 and 1
df_impfct_norm = df_impfct.copy()
df_impfct_norm[VARIABLE_NAMES] = (df_impfct[VARIABLE_NAMES] - df_impfct[VARIABLE_NAMES].min()) / (df_impfct[VARIABLE_NAMES].max() - df_impfct[VARIABLE_NAMES].min())


# Make imperfection score
dcols = [f"{v}" for v in VARIABLE_NAMES]
node_score = (
    df_impfct_norm.groupby("Node Number")[dcols]
     .apply(lambda g: np.square(g.to_numpy()).sum())
     .rename("score")
     .reset_index()
)

df_impfct_norm = df_impfct_norm.groupby(
    ["Node Number", "season", "load", "health", "train_config"]
)[VARIABLE_NAMES].sum().reset_index()

NN_impfct = df_impfct["Node Number"]
df_impfct.info(), NN_impfct.unique()
df_diff

In [ ]:
s = node_score["score"].to_numpy()
med = np.nanmedian(s)
mad = np.nanmedian(np.abs(s - med)) + 1e-12
node_score["z"] = (node_score["score"] - med) / mad
suspect = node_score[node_score["z"] > 10].sort_values("z", ascending=False)
suspect

In [ ]:
# plot_bridge_3d_structure(highlight_nodes=NN_impfct.unique().tolist())
plot_bridge_3d_structure(color_scale=dict(zip(suspect["Node Number"], suspect["score"])), s=3)

In [ ]:
df_impfct[var_cols].mean()

In [ ]:
df_impfct.plot(y="EquivalentStress",kind="hist",bins=50,figsize=(10,5),title="Histogram of Equivalent Stress Differences between two configurations")

# Figure out bridge structure

NOTE: This is now obsolete because we have the node locations.

In [ ]:
len(NODE_NUMBERS)

In [ ]:
from ipywidgets import interact, IntSlider

def integer_factors(n):
    """Returns the list of integer factors of n."""
    factors = []
    for i in range(1, n + 1):
        if n % i == 0:
            factors.append(i)
    return factors
def plot_integer_widths(node_numbers_of_interest: list[int]):
    """Plots the bridge structure as images for all possible widths."""
    img_widths = integer_factors(len(NODE_NUMBERS))
    for w in img_widths:
        img = np.zeros((w,len(NODE_NUMBERS)//w))
        flat_img = img.flatten()
        for i,node_number in enumerate(NODE_NUMBERS):
            if node_number in node_numbers_of_interest:
                flat_img[i] = 1
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='gray', interpolation='nearest')
        plt.title(f'Node Variation Map (Width: {w})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
def plot_load_by_widths_interactive(df, timestep):
    img_widths = integer_factors(len(NODE_NUMBERS))
    times = df["time"].unique()
    if timestep not in times:
        raise ValueError(f"Timestep {timestep} not found in data. Available times: {times}")
    df_time = df[df["time"] == timestep]
    def plot_at_width(width_idx):
        w = img_widths[width_idx]
        img = np.zeros((w, len(NODE_NUMBERS)//w))
        flat_img = img.flatten()
        for i, node_number in enumerate(NODE_NUMBERS):
            load_value = df_time[df_time["Node Number"] == node_number]["TotalDeformation"].values
            assert len(load_value) <= 1, f"Multiple load values found for node {node_number} at time {timestep}"
            if len(load_value) == 1:
                flat_img[i] = load_value[0]
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        plt.title(f'Bridge Load Map at time {timestep} (Width: {w}), (Height: {len(NODE_NUMBERS)//w})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
    interact(plot_at_width, width_idx=IntSlider(min=0, max=len(img_widths)-1, step=1, value=0, description='Width Index'))

def plot_load_by_time_interactive(df, width):
    times = np.sort(df["time"].unique())
    def plot_at_time(time_idx):
        time = times[time_idx]
        df_time = df[df["time"] == time]
        img = np.zeros((width, len(NODE_NUMBERS)//width))
        flat_img = img.flatten()
        for i, node_number in enumerate(NODE_NUMBERS):
            load_value = df_time[df_time["Node Number"] == node_number]["TotalDeformation"].values
            assert len(load_value) <= 1, f"Multiple load values found for node {node_number} at time {time}"
            if len(load_value) == 1:
                flat_img[i] = load_value[0]
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        plt.title(f'Bridge Load Map at time {time} (Width: {width})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
    interact(plot_at_time, time_idx=IntSlider(min=0, max=len(times)-1, step=1, value=0, description='Time Index'))


plot_load_by_widths_interactive(df_sorted[df_sorted["health"]==0], timestep=0.1)
plot_load_by_time_interactive(df_sorted[df_sorted["health"]==0], width=34)

In [ ]:
2210/(42*3)

In [ ]:
plot_integer_widths(comp_diff_node_numbers)

#1#

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import pandas as pd

def animate_3d_structure_data(df, node_xyz, variable_names,
                              title_prefix="Data Evolution",
                              cbar_label="Normalized Value",
                              vmin=None, vmax=None):
    """
    一个通用的 3D 动画函数，可以可视化任何符合格式的 DataFrame。

    Args:
        df (pd.DataFrame): 数据源 (必须包含 'time', 'Node Number' 和 variable_names).
        node_xyz (pd.DataFrame): 坐标源.
        variable_names (list): 要可视化的变量列表 (会自动计算平方和).
        title_prefix (str): 图表标题的前缀.
        cbar_label (str): 颜色条的标签.
        vmin, vmax (float): 手动指定颜色范围 (如果不指定，则自动根据数据最大最小值设定).
    """
    print(f"--- 正在准备动画: {title_prefix} ---")

    # 1. 自动计算全局范围 (如果没有手动指定)
    # 这确保了动画中颜色的一致性
    if vmin is None or vmax is None:
        # 简单估算：基于归一化后的平方和逻辑，范围通常是 0 到 1 (如果是归一化数据)
        # 或者我们在这里不进行归一化，直接画原始数值的平方和？
        # 为了通用性，这里保持之前的逻辑：先归一化，再平方求和
        pass

    # 预计算归一化参数
    norm_params = {}
    for var in variable_names:
        v_min_val = df[var].min()
        v_max_val = df[var].max()
        denom = v_max_val - v_min_val if (v_max_val - v_min_val) != 0 else 1.0
        norm_params[var] = (v_min_val, denom)

    time_steps = np.sort(df['time'].unique())
    print(f"检测到 {len(time_steps)} 个时间步.")

    # 2. 初始化绘图
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.view_init(elev=25, azim=-50)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')

    # 背景结构 (浅灰色)
    ax.scatter(
        node_xyz['X Location (m)'], node_xyz['Y Location (m)'], node_xyz['Z Location (m)'],
        c='lightgray', s=1, alpha=0.1
    )

    scat_plot = [None]

    # 3. 更新函数
    def update(frame_idx):
        t = time_steps[frame_idx]
        df_t = df[df['time'] == t].copy()

        # 计算显示值 (Value Calculation)
        # 逻辑：归一化 -> 平方 -> 求和
        for var in variable_names:
            v_min_val, denom = norm_params[var]
            df_t[var] = (df_t[var] - v_min_val) / denom

        # 按节点汇总 (处理每个节点多个变量的情况)
        node_values = df_t.groupby("Node Number")[variable_names].apply(
            lambda g: np.sum(np.square(g.values))
        ).reset_index(name="plot_val")

        # 合并坐标
        plot_data = node_xyz.merge(node_values, on="Node Number", how="inner")

        # 过滤掉极小值 (仅为了视觉清晰)
        active_nodes = plot_data[plot_data["plot_val"] > 1e-3]
        max_val_now = active_nodes["plot_val"].max() if not active_nodes.empty else 0

        # 清除上一帧
        if scat_plot[0] is not None:
            scat_plot[0].remove()

        # 绘制
        if not active_nodes.empty:
            scat_plot[0] = ax.scatter(
                active_nodes['X Location (m)'],
                active_nodes['Y Location (m)'],
                active_nodes['Z Location (m)'],
                c=active_nodes['plot_val'],
                cmap='plasma',
                vmin=0, vmax=1.0, # 归一化后的数据范围通常在 0-1 之间
                s=30, alpha=1.0, depthshade=False
            )
        else:
            scat_plot[0] = ax.scatter([], [], [], c=[])

        ax.set_title(f"{title_prefix}\nTime: {t:.2f}s | Max Value: {max_val_now:.4f}")
        return scat_plot[0],

    # 添加颜色条
    dummy = ax.scatter([], [], [], c=[], cmap='plasma', vmin=0, vmax=1.0)
    fig.colorbar(dummy, ax=ax, shrink=0.6, label=cbar_label)

    # 生成
    print("正在渲染动画...")
    anim = FuncAnimation(fig, update, frames=len(time_steps), interval=200, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

In [ ]:
anim1 = animate_3d_structure_data(
    df=df_impfct,
    node_xyz=node_xyz,
    variable_names=["EquivalentStress"],
    title_prefix="Imperfection Score (Difference)",
    cbar_label="Norm. Squared Difference"
)
display(anim1)

# Training

In [ ]:
df_train = df_varying.copy()

In [ ]:
X_raw, y_raw = df_train.drop(columns=["health"]), df_train["health"]
X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
    X_raw, y_raw, test_size = 0.1, random_state = 0, shuffle=True,
)

In [ ]:
def standardize(X_train_raw, X_test_raw):
    scaler = sklearn.preprocessing.StandardScaler()

    scaler.fit(X_train_raw)
    X_train = scaler.transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    return X_train, X_test

X_train, X_test = standardize(X_train_raw, X_test_raw)

Decision Tree

In [ ]:
# model = sklearn.tree.DecisionTreeClassifier()
# model.fit(X_train, y_train)

In [ ]:
# model.score(X_test, y_test), model.score(X_train, y_train)

Neural Network

In [ ]:
# model = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(1000,1000,1000),verbose=1)
# model.fit(X_train, y_train)

In [ ]:
# model.score(X_test, y_test), model.score(X_train, y_train)